# Module 6: Improving RAG

**Duration:** 50 minutes

In Module 5 you measured the baseline and saw where it fails. Now we try techniques that
target those failures, and we measure every one of them with the same gold set and the same
`recall_at_k`. Each technique is written as a retrieval function `(query, k) -> chunks`, so it can
be plugged into the evaluation directly.

**By the end of this module you can:**
- add a re-ranking step to an existing pipeline and measure its effect
- explain multi-query retrieval and Reciprocal Rank Fusion
- decide, based on measurements, whether an improvement is worth its extra latency and cost
- argue for or against a technique in plain language

---

## Setup

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import json
import time
from os import getenv
from urllib.parse import urljoin

import requests

from ragsst.evaluation import load_gold_dataset, per_question_hits, recall_at_k
from ragsst.ragtool import RAGTool

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')
MODEL = 'llama3.2:3b'


def generate(prompt: str, temp: float = 0.3) -> str:
    r = requests.post(
        OLLAMA_URL + '/generate',
        json={'model': MODEL, 'prompt': prompt, 'stream': False,
              'options': {'temperature': temp}}
    )
    return json.loads(r.text).get('response', '')


tool = RAGTool(data_path='../data/sample_docs', collection_name='workshop_docs')
tool.make_collection('../data/sample_docs', 'workshop_docs')
gold_dataset = load_gold_dataset('../data/gold_dataset.json')


def baseline_retrieve(query: str, k: int) -> list[str]:
    return tool.collection.query(query_texts=[query], n_results=k)['documents'][0]


results = {}


def score(name: str, retrieve) -> None:
    """Score a retrieval function on the gold set and remember the result."""
    retrieve(gold_dataset[0]['question'], 3)  # warm-up, so model loading is not timed
    start = time.perf_counter()
    for item in gold_dataset:
        retrieve(item['question'], 3)
    latency_ms = (time.perf_counter() - start) / len(gold_dataset) * 1000
    r1 = recall_at_k(retrieve, gold_dataset, k=1)
    r3 = recall_at_k(retrieve, gold_dataset, k=3)
    results[name] = {'Recall@1': r1, 'Recall@3': r3, 'ms/query': latency_ms}
    print(f'{name:<20} Recall@1={r1:.2f}  Recall@3={r3:.2f}  ~{latency_ms:.0f} ms/query')


def changed_questions(retrieve, k: int = 1) -> None:
    """Show which gold questions a technique fixed or broke compared to the baseline."""
    base = per_question_hits(baseline_retrieve, gold_dataset, k)
    new = per_question_hits(retrieve, gold_dataset, k)
    for b, n in zip(base, new):
        if b['hit'] != n['hit']:
            print(f"{'fixed ' if n['hit'] else 'BROKEN'}  {b['question']}")


score('baseline', baseline_retrieve)

These numbers should match the baseline you wrote down in Module 5.

---

## Technique 1: Re-ranking

Embedding models used for retrieval are **bi-encoders**: they encode the query and each chunk
*independently*. That makes them fast (chunk vectors are computed once, at indexing time), but
the model never sees the query and the chunk together.

A **cross-encoder** reads the *pair* (query, chunk) at once and outputs a relevance score.
It is much more accurate, but too slow to run against every chunk in the database.

The standard pattern combines both:
1. Use the bi-encoder to fetch the top N candidates (e.g. N=10).
2. Re-score only those candidates with the cross-encoder.
3. Keep the best k.

Hypothesis from Module 5: many misses had the right chunk in the top 3 but not at rank 1.
Re-ranking should help exactly there.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


def rerank_retrieve(query: str, k: int, candidates: int = 10) -> list[str]:
    docs = baseline_retrieve(query, candidates)
    scores = reranker.predict([[query, doc] for doc in docs])
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:k]]


score('re-ranking', rerank_retrieve)
print()
changed_questions(rerank_retrieve)

Look at a fixed question side by side. What did the cross-encoder see that the bi-encoder missed?

In [ ]:
def compare_top(question: str, k: int = 1, **retrievers) -> None:
    print(f'Q: {question}\n')
    for name, retrieve in retrievers.items():
        for chunk in retrieve(question, k):
            print(f"{name:<12} {chunk.replace(chr(10), ' ')[:200]}...")
        print()


compare_top('Can I take the train in first class?',
            baseline=baseline_retrieve, reranked=rerank_retrieve)

**Check the cost:** compare the `ms/query` of the baseline and re-ranking. Would that extra
latency matter for a chatbot? For a nightly batch job? Also note that re-ranking can *break*
questions that worked before: a better average does not mean better for every question.

---

## Technique 2: RAG-Fusion (multi-query + Reciprocal Rank Fusion)

A single query searches in a single direction of the embedding space. If the document uses
different words ("distributed working arrangement" instead of "work from home"), the right chunk
may not come up.

**Multi-query** asks the LLM to rephrase the question several times and retrieves for each
version. **RAG-Fusion** then merges the ranked lists with **Reciprocal Rank Fusion (RRF)**: each
chunk gets a score of $\sum_{l} \frac{1}{k + r}$ over all lists $l$ in which it appears at
rank $r$, with $k = 60$. A chunk that ranks high for several phrasings wins. RRF uses ranks
rather than raw similarity scores, so it does not matter that different queries produce scores on
different scales.

> This technique needs one LLM call per question. We cache the rephrasings, so the 12 gold
> questions cost 12 calls: a few seconds with a GPU, one to five minutes on a laptop CPU.

In [ ]:
_variant_cache: dict[str, list[str]] = {}


def generate_query_variants(query: str, n: int = 3) -> list[str]:
    if query not in _variant_cache:
        prompt = (
            f'Write {n} different versions of the following question. '
            'Each version should ask for the same information but use different words. '
            'Output only the questions, one per line, with no numbering or extra text.\n\n'
            f'Question: {query}'
        )
        response = generate(prompt, temp=0.7)
        lines = [line.strip() for line in response.strip().split('\n') if line.strip()]
        _variant_cache[query] = lines[:n]
    return _variant_cache[query]


def reciprocal_rank_fusion(ranked_lists: list[list[str]], k: int = 60) -> list[str]:
    scores: dict[str, float] = {}
    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return [doc for doc, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]


def fusion_retrieve(query: str, k: int, n_per_query: int = 5) -> list[str]:
    queries = [query] + generate_query_variants(query)
    ranked_lists = [baseline_retrieve(q, n_per_query) for q in queries]
    return reciprocal_rank_fusion(ranked_lists)[:k]


print('Rephrasings of one question:')
for v in generate_query_variants('Who pays for my home internet?'):
    print(' -', v)

In [ ]:
score('rag-fusion', fusion_retrieve)
print()
changed_questions(fusion_retrieve)

Now run it a second time with fresh rephrasings:

In [ ]:
_variant_cache.clear()
score('rag-fusion (run 2)', fusion_retrieve)

Did you get the same numbers? In our test runs RAG-Fusion matched the baseline in one run and
beat it in the next. When part of the pipeline is an LLM sampling at temperature 0.7, the
*measurement itself* is noisy, and with 12 questions one question more or less changes the score
by 0.08. Before you believe a difference, run it several times and use more questions.

---

## Technique 3 (optional, if time): HyDE

Questions and answers look different: "What is the per diem?" does not look like
"The daily allowance for meals is 28 euros." **HyDE** (Hypothetical Document Embeddings) asks
the LLM to write a *fake answer* first, and searches with that instead of the question.
A fake answer looks like an answer, so it should land near real answers in embedding space.

It is high-variance: when the fake answer is wrong in the right *style*, it works well; when it
invents the wrong topic, retrieval goes in the wrong direction. It costs one LLM call per query.

In [ ]:
_hyde_cache: dict[str, str] = {}


def hyde_retrieve(query: str, k: int) -> list[str]:
    if query not in _hyde_cache:
        prompt = (
            'Write a short, factual passage that would directly answer the following question. '
            'Do not say you are guessing. '
            'Just write the passage as if it were from a document.\n\n'
            f'Question: {query}\n'
            'Passage:'
        )
        _hyde_cache[query] = generate(prompt, temp=0.5)
    return baseline_retrieve(_hyde_cache[query], k)


score('hyde', hyde_retrieve)
print()
changed_questions(hyde_retrieve)

---

## Summary: decide and explain

In [ ]:
print(f"{'technique':<20} {'Recall@1':>9} {'Recall@3':>9} {'ms/query':>9}")
print('-' * 50)
for name, r in results.items():
    print(f"{name:<20} {r['Recall@1']:>9.2f} {r['Recall@3']:>9.2f} {r['ms/query']:>9.0f}")

| If this is failing | Try this |
|-------------------|----------|
| Right chunk is in the top 3 but not at rank 1 | Re-ranking |
| Right chunk is not retrieved at all; vocabulary mismatch | RAG-Fusion, HyDE, a better embedding model |
| Outdated or duplicate documents win | Fix the data (Module 5, exercise 3) |
| The LLM ignores the context | Prompt engineering (Module 4) |

**Decide and explain (10 minutes, in pairs):** Nordlicht wants to launch an internal HR chatbot
on the handbook. Which technique would you ship, if any? Write a recommendation of three
sentences for a manager who does not know what an embedding is: what improves, what it costs
(latency, complexity, LLM calls), and how sure you are given 12 test questions.

### From chunks to answers

So far we only looked at retrieval. Here is what the improvement means for the final answer:

In [ ]:
question = 'Can I take the train in first class?'
for name, retrieve in [('baseline', baseline_retrieve), ('re-ranking', rerank_retrieve)]:
    context = '\n\n'.join(retrieve(question, 3))
    print(f'{name}: {generate(tool.get_context_prompt(question, context))}\n')

---

**Exercises**

1. Combine both: take the RAG-Fusion candidates (top 10) and re-rank them with the
   cross-encoder. Score it. Is it better than each technique alone? What does it cost?

2. Change `candidates` in `rerank_retrieve` to 5 and to 25. How do Recall@1 and latency change?

3. *Hard:* generate several hypothetical answers per question and fuse their result lists with
   RRF (HyDE + RAG-Fusion). Does it beat both on Recall@1? Run it twice.

---

**Further reading**

- Cross-encoders vs bi-encoders: https://www.sbert.net/docs/cross_encoder/usage.html
- RRF original paper: https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf
- RAG-Fusion: https://arxiv.org/abs/2402.03367
- HyDE paper: https://arxiv.org/abs/2212.10496